# Feasibility frontier (three-objective analysis)

Stage 5 of the MOF-ECG pipeline. Joins bootstrap $f_P$, $f_S$ with XAI $f_E$, then plots the feasibility frontier in $(K_S, K_E)$ threshold space and reports reliable shadow prices.

**Configuration:** same YAML / `MOF_DATA_ROOT` as training.

**Prerequisites:**
1. `hyper_sweep/bootstrap_results.csv` (notebook `01_bootstrap_stability.ipynb` or `python src/mof_analysis.py bootstrap`)
2. `run_xai.py --hyper_sweep_fe` outputs under `xai_results/hyper_sweep_fe/`

Per-code Slurm XAI jobs write `sweep_fe_<CODE>.csv`; this notebook auto-merges them into `sweep_fe_summary.csv` when needed.


In [ ]:
import os

In [ ]:
import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
import pandas as pd

_REPO = Path.cwd().resolve()
if (_REPO / "src" / "mof_analysis.py").is_file():
    SRC = _REPO / "src"
elif (_REPO.parent / "src" / "mof_analysis.py").is_file():
    SRC = _REPO.parent / "src"
else:
    raise FileNotFoundError("Cannot find src/mof_analysis.py")

if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from mof_analysis import (
    compute_feasibility_frontier,
    is_feasible,
    load_analysis_config,
    load_three_objective_dataset,
    merge_sweep_fe_csvs,
    reliable_shadow_price,
    resolve_paths,
)

%matplotlib inline
plt.rcParams.update({
    "figure.dpi": 130,
    "font.family": "DejaVu Sans",
    "axes.spines.top": False,
    "axes.spines.right": False,
})

# Jupyter kernels do not inherit `source configs/env.ecgpsych`
_REPO_ROOT = SRC.parent
os.environ.setdefault("MOF_CONFIG", str(_REPO_ROOT / "configs" / "ecgpsych.yaml"))
os.environ.setdefault("MOF_DATA_ROOT", str(_REPO_ROOT / "artifacts" / "runs"))
os.environ.setdefault("MOF_PREBUILT_ROOT", str(_REPO_ROOT / "artifacts" / "prebuilt"))

CONFIG_PATH = os.environ.get("MOF_CONFIG")

cfg = load_analysis_config(CONFIG_PATH)
paths = resolve_paths(cfg)
print("config:", CONFIG_PATH)
print("data_root:", paths["data_root"])
print("codes:", cfg.diagnoses.codes)

RECALL_MIN = 0.30   # drop configs with very low case recall
N_GRID = 80
N_MIN_FEAS = 3      # reliability criterion for shadow prices

# Optional clinical requirement levels per code (edit for your study)
REQUIREMENTS = {code: (0.50, 0.40) for code in cfg.diagnoses.codes}

merge_sweep_fe_csvs(cfg)


In [ ]:
df = load_three_objective_dataset(cfg, recall_min=RECALL_MIN)
print(f"Configurations after recall >= {RECALL_MIN}: {len(df)}")
print("Per-code counts:")
print(df.groupby("psych_code").size().rename("N_configs"))
for col in ["f_P", "f_S", "f_E"]:
    print(f"  {col}: {df[col].min():.3f} – {df[col].max():.3f}")
df.head()


In [ ]:
codes = [c for c in cfg.diagnoses.codes if c in set(df["psych_code"])]
ncols = min(3, max(1, len(codes)))
nrows = max(1, (len(codes) + ncols - 1) // ncols)
fig, axes = plt.subplots(nrows, ncols, figsize=(6 * ncols, 5 * nrows), squeeze=False)

frontier_results = {}
for ax, code in zip(axes.flatten(), codes):
    sub = df[df["psych_code"] == code].copy()
    if sub.empty:
        ax.set_title(f"{code}\n(no data)")
        continue

    frontier_df, pareto = compute_feasibility_frontier(sub, n_grid=N_GRID)
    frontier_results[code] = {"frontier_df": frontier_df, "pareto": pareto}

    ku_vals = np.linspace(sub["f_S"].min(), sub["f_S"].max(), 50)
    ke_vals = np.linspace(sub["f_E"].min(), sub["f_E"].max(), 50)
    KS, KE = np.meshgrid(ku_vals, ke_vals)
    feasmap = np.array([[1 if is_feasible(sub, ks, ke) else 0 for ks in ku_vals] for ke in ke_vals], dtype=float)
    ax.contourf(KS, KE, feasmap, levels=[-0.5, 0.5, 1.5], colors=["#FFCDD2", "#C8E6C9"], alpha=0.45)

    if len(pareto) > 1:
        ax.plot(pareto["K_S"], pareto["max_K_E"], "k-", lw=2.5, zorder=6)
        ax.plot(pareto["K_S"], pareto["max_K_E"], "ko", ms=4, zorder=7)

    sc = ax.scatter(sub["f_S"], sub["f_E"], c=sub["f_P"], cmap="RdYlGn", s=45, alpha=0.85, vmin=0, vmax=1, zorder=5)

    if len(pareto) > 0:
        tmp = pareto.copy()
        tmp["sum"] = tmp["K_S"] + tmp["max_K_E"]
        best_point = tmp.loc[tmp["sum"].idxmax()]
        bx, by = float(best_point["K_S"]), float(best_point["max_K_E"])
        ax.plot(bx, by, "*", ms=14, color="#1565C0", zorder=8)
        ax.annotate(f"$K_S$={bx:.2f}\n$K_E$={by:.2f}", xy=(bx, by), fontsize=9, color="#1565C0",
                    bbox=dict(boxstyle="round,pad=0.3", fc="white", ec="#1565C0", alpha=0.9))

    ax.set_title(code, fontweight="bold")
    ax.set_xlabel("$f_S$ (stability)")
    ax.set_ylabel("$f_E$ (explainability)")
    ax.grid(True, alpha=0.2)

for ax in axes.flatten()[len(codes):]:
    ax.axis("off")

cbar = fig.colorbar(sc, ax=axes.ravel().tolist(), fraction=0.02, pad=0.02)
cbar.set_label("$f_P$")
fig.suptitle("Feasibility frontier per diagnosis code", y=1.02)
fig.tight_layout()
# plot_path = paths["analysis_dir"] / "feasibility_frontier.png"
# paths["analysis_dir"].mkdir(parents=True, exist_ok=True)
# fig.savefig(plot_path, dpi=150, bbox_inches="tight")
# print(f"Saved: {plot_path}")
plt.show()


In [ ]:
rows = []
for code in codes:
    sub = df[df["psych_code"] == code]
    if sub.empty:
        continue
    k_s, k_e = REQUIREMENTS.get(code, (sub["f_S"].median(), sub["f_E"].median()))
    result = reliable_shadow_price(sub, k_s, k_e, n_min=N_MIN_FEAS)
    if result is None:
        rows.append({"psych_code": code, "K_S": k_s, "K_E": k_e, "status": "INFEASIBLE"})
        continue
    rows.append({
        "psych_code": code,
        "K_S": k_s,
        "K_E": k_e,
        "fP_star": result["fP_star"],
        "lambda_S": result["lambda_S"],
        "lambda_E": result["lambda_E"],
        "reliable_S": result["reliable_S"],
        "reliable_E": result["reliable_E"],
        "n_feasible": result["n_feasible"],
        "status": "OK",
    })

shadow_df = pd.DataFrame(rows)
# shadow_path = paths["analysis_dir"] / "shadow_prices.csv"
# shadow_df.to_csv(shadow_path, index=False)
# print(f"Saved: {shadow_path}")
display(shadow_df)


In [ ]:
# Threshold sweep: f_P* vs K_E (K_S fixed at joint max-sum) — same style as
# jo_trial6_data_exploration/12threshold_sweep_v3.ipynb
from scipy.ndimage import uniform_filter1d

N_SWEEP_POINTS = N_GRID  # 80

def sweep_fP_vs_threshold(df_code, sweep_col, anchor_col, anchor_val, n_points=N_SWEEP_POINTS):
    """Sweep threshold on sweep_col with anchor_col held at >= anchor_val."""
    K_vals = np.linspace(df_code[sweep_col].min(), df_code[sweep_col].max(), n_points)
    records = []
    for k in K_vals:
        if sweep_col == "f_E":
            feas = df_code[(df_code["f_E"] >= k) & (df_code["f_S"] >= anchor_val)]
        else:
            feas = df_code[(df_code["f_S"] >= k) & (df_code["f_E"] >= anchor_val)]
        if len(feas) > 0:
            best = feas.loc[feas["f_P"].idxmax()]
            records.append({
                "K": k,
                "fP_star": float(best["f_P"]),
                "N_feas": len(feas),
                "feasible": True,
                "best_fS": float(best["f_S"]),
                "best_fE": float(best["f_E"]),
            })
        else:
            records.append({
                "K": k,
                "fP_star": np.nan,
                "N_feas": 0,
                "feasible": False,
                "best_fS": np.nan,
                "best_fE": np.nan,
            })
    return pd.DataFrame(records)


def compute_slope(sweep_df, smooth_window=3):
    """Central finite-difference d(fP*)/dK; shadow_price = max(0, -slope)."""
    out = sweep_df.copy()
    fP = out["fP_star"].values
    K = out["K"].values
    n = len(out)
    slope = np.full(n, np.nan)
    for i in range(1, n - 1):
        if not np.isnan(fP[i - 1]) and not np.isnan(fP[i + 1]):
            slope[i] = (fP[i + 1] - fP[i - 1]) / (K[i + 1] - K[i - 1])
    if n >= 2 and not np.isnan(fP[0]) and not np.isnan(fP[1]):
        slope[0] = (fP[1] - fP[0]) / (K[1] - K[0])
    if n >= 2 and not np.isnan(fP[-1]) and not np.isnan(fP[-2]):
        slope[-1] = (fP[-1] - fP[-2]) / (K[-1] - K[-2])
    if smooth_window > 1:
        valid = ~np.isnan(slope)
        if valid.sum() > smooth_window:
            slope_filled = np.where(valid, slope, 0.0)
            slope_smooth = uniform_filter1d(slope_filled, size=smooth_window)
            slope = np.where(valid, slope_smooth, np.nan)
    out["slope"] = slope
    out["shadow_price"] = np.where(~np.isnan(slope), np.maximum(0.0, -slope), np.nan)
    out["reliable"] = out["N_feas"] >= N_MIN_FEAS
    return out


def find_infeasibility_onset(sweep_df):
    feas = sweep_df[sweep_df["feasible"]]
    infeas = sweep_df[~sweep_df["feasible"]]
    last_feasible = float(feas["K"].max()) if len(feas) > 0 else np.nan
    first_infeas = float(infeas["K"].min()) if len(infeas) > 0 else np.nan
    return last_feasible, first_infeas


# Joint max-sum anchors from the feasibility frontier (same definition as MIMIC)
JOINT_MAX = {}
for code in codes:
    pareto = frontier_results[code]["pareto"]
    if len(pareto) == 0:
        continue
    tmp = pareto.copy()
    tmp["sum"] = tmp["K_S"] + tmp["max_K_E"]
    best = tmp.loc[tmp["sum"].idxmax()]
    JOINT_MAX[code] = (float(best["K_S"]), float(best["max_K_E"]))
    print(f"{code}: K_S={JOINT_MAX[code][0]:.4f}, K_E={JOINT_MAX[code][1]:.4f}")

Y_MIN, Y_MAX = -0.05, 1.10
# Scenario lines from the NeurIPS paper (optional reference overlays)
OLD_SCENARIOS = {
    "Lenient": {"K_S": 0.25, "K_E": 0.42},
    "Moderate": {"K_S": 0.50, "K_E": 0.45},
    "Strict": {"K_S": 0.75, "K_E": 0.50},
}
SCEN_COLORS = {
    "Lenient": "#4CAF50",
    "Moderate": "#FF9800",
    "Strict": "#F44336",
}
K_E_FLOOR = 0.42

CODE = "SCZ"
sub = df[df["psych_code"] == CODE].copy()
ks_anchor, ke_anchor = JOINT_MAX[CODE]

sweep = sweep_fP_vs_threshold(
    sub, sweep_col="f_E", anchor_col="f_S", anchor_val=ks_anchor
)
sweep = compute_slope(sweep)
last_feas, first_inf = find_infeasibility_onset(sweep)

fig, ax = plt.subplots(1, 1, figsize=(8.5, 5.5))

feas_mask = sweep["feasible"]
ax.plot(
    sweep.loc[feas_mask, "K"],
    sweep.loc[feas_mask, "fP_star"],
    color="#1565C0",
    lw=2.8,
    zorder=5,
    label=r"$f_P^*$",
)

if not np.isnan(first_inf):
    ax.axvspan(first_inf, sweep["K"].max(), color="#FFCDD2", alpha=0.45, zorder=1)
    ax.axvline(
        last_feas,
        color="#C62828",
        lw=1.8,
        ls="--",
        alpha=0.9,
        zorder=4,
        label=f"Frontier ($K_E={last_feas:.2f}$)",
    )

k_min, k_max = float(sweep["K"].min()), float(sweep["K"].max())
if k_min <= K_E_FLOOR <= k_max:
    ax.axvline(K_E_FLOOR, color="black", lw=1.2, ls=":", alpha=0.55, zorder=3)

for scen_name, scen_vals in OLD_SCENARIOS.items():
    k_val = scen_vals["K_E"]
    if k_min <= k_val <= k_max:
        ax.axvline(
            k_val,
            color=SCEN_COLORS[scen_name],
            lw=1.2,
            ls=":",
            alpha=0.75,
            label=scen_name,
        )

ax2 = ax.twinx()
ax2.fill_between(sweep["K"], sweep["N_feas"], alpha=0.10, color="gray", zorder=0)
ax2.set_ylabel(r"$N_{\mathrm{feas}}$", fontsize=11, color="gray")
ax2.tick_params(axis="y", labelcolor="gray", labelsize=10)
ax2.set_ylim(bottom=0)

ax.text(
    0.97,
    0.97,
    "Psychiatry-ECG: SCZ vs other psych",
    transform=ax.transAxes,
    fontsize=11,
    fontweight="bold",
    color="#1565C0",
    ha="right",
    va="top",
    bbox=dict(boxstyle="round,pad=0.35", facecolor="white", edgecolor="#1565C0", alpha=0.90),
)

ax.set_title(
    f"{CODE}  |  $f_P^*$ vs $K_E$  ($K_S$ fixed at {ks_anchor:.2f})",
    fontsize=12,
    fontweight="bold",
    pad=10,
)
ax.set_xlabel(r"$K_E$ (explainability threshold)", fontsize=16)
ax.set_ylabel(r"$f_P^*$ (best feasible performance)", fontsize=16)
ax.set_ylim(Y_MIN, Y_MAX)
ax.set_xlim(k_min, k_max)
ax.tick_params(labelsize=11)
ax.grid(True, alpha=0.18, lw=0.6)
ax.legend(fontsize=10, loc="lower left", framealpha=0.88, ncol=2, borderpad=0.6, handlelength=1.8)

fig.tight_layout()
paths["analysis_dir"].mkdir(parents=True, exist_ok=True)
plot_path = paths["analysis_dir"] / "threshold_sweep_fP_vs_KE_SCZ.png"
fig.savefig(plot_path, dpi=150, bbox_inches="tight")
print(f"Saved: {plot_path}")
print(f"last feasible K_E={last_feas:.4f}, first infeasible K_E={first_inf}")
plt.show()
